# Évaluation sur GPU (Kaggle)

Ce notebook exécute les évaluations du projet sur un GPU Kaggle, avec exactement le même code que la version locale :

1. installation d'Ollama et téléchargement des modèles ;
2. récupération du dépôt GitHub et installation du package ;
3. construction de l'index vectoriel ;
4. évaluation et archivage des résultats.

Paramètres Kaggle requis : **Accelerator : GPU T4 x2** et **Internet : On**.

In [ ]:
REPO_URL = "https://github.com/louchenilounes/assistant-rag-agents.git"
RUNS = [("agents", "dev")]
MODELS = ["qwen3:4b-instruct", "bge-m3", "qwen3:8b"]

## 1. Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!ollama --version

In [ ]:
import subprocess
import time
import urllib.request

server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("Serveur Ollama démarré")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Le serveur Ollama n'a pas démarré, voir /kaggle/working/ollama.log")

In [ ]:
for model in MODELS:
    subprocess.run(["ollama", "pull", model], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"{model} téléchargé")
!ollama list

## 2. Projet

In [ ]:
!rm -rf /kaggle/working/assistant-rag-agents
!git clone -q {REPO_URL} /kaggle/working/assistant-rag-agents
%cd /kaggle/working/assistant-rag-agents
!git log -1 --format="%h %s"
!pip install -q -e .

## 3. Index et évaluation

In [ ]:
!python scripts/check_documents.py
!python scripts/build_index.py

In [ ]:
for system, split in RUNS:
    subprocess.run(["python", "scripts/evaluate.py", "--system", system, "--split", split], check=True)

## 4. Résultats

L'archive `resultats.zip` apparaît dans l'onglet **Output** du notebook. Elle se décompresse dans le dossier `results/` du projet.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/resultats", "zip", "results")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv